# PharmaLens AI — Achievement Sheet Engine
Calculates monthly, YTD and annual achievement from actual sales + targets.

**Overall Value Achievement % = Total Actual Sales Value ÷ Total Target Value × 100**

Never average product achievement percentages for the overall KPI.


In [ ]:
import pandas as pd
import numpy as np

DEFAULT_GREEN=90.0
DEFAULT_YELLOW=80.0

def status_color(pct,green=DEFAULT_GREEN,yellow=DEFAULT_YELLOW):
    if pd.isna(pct): return "N/A"
    if pct>=green: return "GREEN"
    if pct>=yellow: return "YELLOW"
    return "RED"

def ensure_sales_value(df):
    df=df.copy()
    if "units" in df.columns: df["units"]=pd.to_numeric(df["units"],errors="coerce")
    if "sales_value" not in df.columns:
        df["sales_value"]=np.nan
    df["sales_value"]=pd.to_numeric(df["sales_value"],errors="coerce")
    if "unit_price" in df.columns:
        df["unit_price"]=pd.to_numeric(df["unit_price"],errors="coerce")
        mask=df["sales_value"].isna() & df["units"].notna() & df["unit_price"].notna()
        df.loc[mask,"sales_value"]=df.loc[mask,"units"]*df.loc[mask,"unit_price"]
    return df


## Product achievement
Partial targets are supported. Missing targets are excluded, not zeroed.


In [ ]:
def achievement_table(sales,targets):
    sales=ensure_sales_value(sales); targets=targets.copy()
    actual=sales.groupby("product",dropna=False).agg(
        actual_units=("units","sum"),actual_value=("sales_value","sum")).reset_index()
    agg={}
    if "target_units" in targets.columns: agg["target_units"]=("target_units","sum")
    if "target_value" in targets.columns: agg["target_value"]=("target_value","sum")
    if agg:
        tgt=targets.groupby("product",dropna=False).agg(**agg).reset_index()
        out=actual.merge(tgt,on="product",how="left")
    else:
        out=actual.copy(); out["target_units"]=np.nan; out["target_value"]=np.nan
    out["unit_achievement_pct"]=np.where(out["target_units"].notna() & (out["target_units"]!=0),
        out["actual_units"]/out["target_units"]*100,np.nan)
    out["value_achievement_pct"]=np.where(out["target_value"].notna() & (out["target_value"]!=0),
        out["actual_value"]/out["target_value"]*100,np.nan)
    out["achievement_status"]=out["value_achievement_pct"].apply(status_color)
    return out


## Overall KPI
Value achievement is the default commercial KPI when a valid target value exists. If only target units exist, report unit achievement only.


In [ ]:
def overall_kpi(ach):
    actual_units=float(ach["actual_units"].sum())
    actual_value=float(ach["actual_value"].sum())
    tu=float(ach["target_units"].sum()) if "target_units" in ach and ach["target_units"].notna().any() else np.nan
    tv=float(ach["target_value"].sum()) if "target_value" in ach and ach["target_value"].notna().any() else np.nan
    return {
        "actual_units":actual_units,"actual_value":actual_value,
        "target_units":tu,"target_value":tv,
        "unit_achievement_pct":actual_units/tu*100 if pd.notna(tu) and tu!=0 else np.nan,
        "value_achievement_pct":actual_value/tv*100 if pd.notna(tv) and tv!=0 else np.nan,
        "status":status_color(actual_value/tv*100 if pd.notna(tv) and tv!=0 else np.nan)
    }


## Monthly / YTD / Annual
Monthly achievement requires monthly target grain. Annual targets must not be silently divided into monthly targets.


In [ ]:
def monthly_achievement(sales,targets):
    sales=ensure_sales_value(sales)
    if "month" not in sales.columns or "month" not in targets.columns:
        return pd.DataFrame({"message":["Monthly target grain unavailable"]})
    a=sales.groupby(["year","month"],dropna=False).agg(
        actual_units=("units","sum"),actual_value=("sales_value","sum")).reset_index()
    kwargs={}
    if "target_units" in targets.columns: kwargs["target_units"]=("target_units","sum")
    if "target_value" in targets.columns: kwargs["target_value"]=("target_value","sum")
    t=targets.groupby(["year","month"],dropna=False).agg(**kwargs).reset_index() if kwargs else pd.DataFrame()
    if t.empty: return pd.DataFrame({"message":["Monthly target unavailable"]})
    out=a.merge(t,on=["year","month"],how="left")
    out["value_achievement_pct"]=np.where(out["target_value"].notna() & (out["target_value"]!=0),
        out["actual_value"]/out["target_value"]*100,np.nan) if "target_value" in out else np.nan
    out["achievement_status"]=out["value_achievement_pct"].apply(status_color)
    return out


## PM/Retail, AM/Hospital and distributor views
These views appear only when the source/channel data exists. The user is not forced to provide these dimensions.


In [ ]:
def source_summary(sales):
    sales=ensure_sales_value(sales)
    col="distributor" if "distributor" in sales.columns else (
        "detected_source" if "detected_source" in sales.columns else (
            "channel" if "channel" in sales.columns else None))
    if not col: return pd.DataFrame()
    out=sales.groupby(col).agg(sales_units=("units","sum"),sales_value=("sales_value","sum")).reset_index()
    total=out["sales_value"].sum()
    out["sales_share_pct"]=np.where(total!=0,out["sales_value"]/total*100,np.nan)
    return out.sort_values("sales_value",ascending=False)

def target_coverage(sales,targets):
    if "product" not in sales.columns or "product" not in targets.columns: return np.nan
    p=set(sales["product"].dropna().astype(str)); t=set(targets["product"].dropna().astype(str))
    return len(p&t)/len(p)*100 if p else np.nan


## No-target behavior
No target is a normal state: switch to Sales Analytics Mode. Do not error and do not invent a target.


In [ ]:
def dashboard_payload(sales,targets=None):
    targets=targets if targets is not None else pd.DataFrame()
    sales=ensure_sales_value(sales)
    if targets.empty or not (
        ("target_value" in targets.columns and targets["target_value"].notna().any()) or
        ("target_units" in targets.columns and targets["target_units"].notna().any())):
        return {
            "mode":"SALES_ANALYTICS_MODE","target_status":"NO_TARGET",
            "message":"Target data not detected. Upload/add target or continue with Sales Analytics.",
            "sales_value":float(sales["sales_value"].sum()),
            "sales_units":float(sales["units"].sum()) if "units" in sales.columns else None
        }
    ach=achievement_table(sales,targets)
    return {"mode":"ACHIEVEMENT_MODE","target_status":"TARGET_AVAILABLE",
            "overall":overall_kpi(ach),
            "target_coverage_pct":target_coverage(sales,targets),
            "product_achievement":ach.to_dict(orient="records")}

EXPORT_SHEETS=[
    "Executive_KPI","Sales_Consolidated","Targets_Consolidated","Product_Achievement",
    "Monthly_Achievement","YTD_Achievement","Channel_Achievement",
    "Distributor_Intelligence","Data_Quality","Dashboard"
]


## Excel export requirements
Use the existing project exporter if available. Otherwise create the workbook with the sheets above, conditional formatting (Green ≥90%, Yellow 80–<90%, Red <80%) and a visible **PharmaLens AI** watermark/header/footer on every exported report.
